# AIML Practical 04 — Classification Models

**SVM · Decision Tree · KNN · Gaussian Naïve Bayes**  
Dataset: Wisconsin Diagnostic Breast Cancer  
Positive class: **1 = malignant**; negative class: **0 = benign**.

**Goal:** Compare four supervised classifiers using leakage-free stratified cross-validation, F1-based hyperparameter search, multiple test metrics, and visualizations.

> Educational experiment only. Not a medical diagnostic system.


## 1. Imports

The notebook reuses the exact functions defined in `classification_models.py`. This prevents discrepancies between the script, notebook, and report, while keeping the implementation accessible in a single module.

In [ ]:
import pandas as pd
from IPython.display import display, Image, Markdown
from pathlib import Path
from classification_models import (
    SEED, OUTPUT_DIR, load_data, split_data, model_specs,
    train_and_evaluate, save_plots
)

print('Random state:', SEED)


## 2. Load and explore the dataset

569 observations, 30 features, and no missing values are expected. The original scikit-learn class numbering is remapped so that malignant is positive.

In [ ]:
X, y = load_data()
print('Dataset shape:', X.shape)
print('Class counts (0 benign; 1 malignant):')
print(y.value_counts().sort_index())
print('Missing entries:', int(X.isna().sum().sum()))
display(X.head(5))


## 3. Stratified train/test split

The held-out 20% test set is not used during model or hyperparameter selection.

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Test class distribution:', y_test.value_counts().sort_index().to_dict())


## 4. Classifiers and hyperparameter grids

SVM and KNN use a `StandardScaler` inside their pipelines. Decision Trees and Gaussian Naïve Bayes use the original numerical features.

In [ ]:
for model_name, (estimator, params) in model_specs().items():
    print(model_name)
    print('  estimator:', estimator)
    print('  grid:', params)


## 5. Five-fold cross-validation and tuning

For every estimator we first measure the untuned CV F1-score and then use `GridSearchCV(scoring='f1')` on **training folds only**. F1 refers to malignant-class F1-score because malignant is encoded as 1.

In [ ]:
results, trained_models = train_and_evaluate(X_train, X_test, y_train, y_test)
fields = ['Model', 'Baseline_CV_F1', 'Tuned_CV_F1', 'CV_F1_SD',
          'Accuracy', 'Precision', 'Recall', 'F1', 'Log_Loss', 'ROC_AUC']
display(results[fields].round(4))


## 6. Selection, confusion counts, and train/test gap

TN and TP are correctly labeled benign and malignant observations respectively. FN is an actual malignant observation incorrectly classified as benign.

In [ ]:
display(results[['Model', 'TN', 'FP', 'FN', 'TP', 'Train_Accuracy', 'Accuracy', 'Best_Params']])
print('Top model by training CV F1:', results.iloc[0]['Model'])


## 7. Export charts and study the figures

Each saved chart is generated from the fitted models and their held-out predictions—not typed-in scores.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
save_plots(results, trained_models, X_test, y_test, OUTPUT_DIR)
results.to_csv(OUTPUT_DIR / 'classification_results.csv', index=False)
for chart in ('metrics_comparison.png', 'confusion_matrices.png', 'roc_curves.png'):
    display(Markdown(f'### {chart}'))
    display(Image(filename=str(OUTPUT_DIR / chart), width=800))


## 8. Findings and reflection

- **SVM** had the strongest test F1-score (0.9630), highest accuracy (0.9737), and lowest log loss (0.0946) in this run.
- **Naïve Bayes** showed slightly higher ROC-AUC, which evaluates ranking across decision thresholds—not classification recall at the default threshold.
- **False negatives matter:** SVM missed three of the 42 malignant cases on the held-out set. No model was perfect.
- **Decision Trees** offer clear decision rules; **KNN** provides a simple neighborhood-based baseline; **Naïve Bayes** is efficient but assumes conditional independence.
- Before any consequential application, validate on independent populations, study subgroup reliability, and investigate thresholds and calibration.

The notebook is intended for learning and reproducibility, not clinical decision-making.